# 05 — Predictive Modeling

Can we predict whether a customer will leave a rating? This is a binary classification problem with a ~61/39 class split. We compare Logistic Regression, Random Forest, and Gradient Boosting using stratified 5-fold cross-validation.

In [ ]:
from pathlib import Path
import pandas as pd
from src.data.load import load_foodhub
from src.features.build import clean_data, engineer_features, encode_categoricals
from src.models.predict import (
    prepare_classification_data,
    evaluate_classifier,
    compare_models,
    get_feature_importances,
    get_roc_data,
)
from src.visualization.plots import (
    plot_feature_importance,
    plot_roc_curve,
)

In [ ]:
df = load_foodhub(Path("data/raw/foodhub_order.csv"))
df = clean_data(df)
df = engineer_features(df)
df_encoded = encode_categoricals(df)
X, y = prepare_classification_data(df_encoded)
print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")
print(f"Class balance: {y.mean():.1%} rated, {1-y.mean():.1%} not rated")

## Baseline

A naive classifier that always predicts the majority class (rated) achieves ~61% accuracy. Our models need to meaningfully beat this.

## Model Comparison

In [ ]:
comparison = compare_models(X, y)
comparison

## ROC Curves

In [ ]:
roc_results = {}
for name in ["Logistic Regression", "Random Forest", "Gradient Boosting"]:
    fpr, tpr, auc = get_roc_data(X, y, model_name=name)
    roc_results[name] = (fpr, tpr, auc)

fig = plot_roc_curve(roc_results)
fig

## Feature Importances

In [ ]:
importances = get_feature_importances(X, y, model_name="Random Forest")
importances.head(15)

In [ ]:
fig = plot_feature_importance(importances, top_n=15)
fig

## Limitations

- **Small dataset** (1,898 rows) limits model complexity and generalizability
- **No temporal features** — we can't model time-of-day or recency effects
- **ROC on training data** is shown for visualization only — all evaluation metrics come from cross-validation
- **Feature engineering is constrained** by the 9 original columns — real-world models would include user history, restaurant metadata, and external signals

## What This Demonstrates

Even with limited data, the modeling pipeline shows:
1. Proper cross-validation methodology (stratified, not random split)
2. Multiple model comparison with consistent evaluation
3. Feature importance analysis for interpretability
4. Honest reporting of limitations alongside results